In [1]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
import numpy as np
import torch

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0813 16:08:35.092000 15528 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
SUBSET = "5k"
DATA_PATH = f"../data/mathcodeinstruct-train-{SUBSET}.json"
EVAL_PATH = f"../data/mathcodeinstruct-eval.json"
OUTPUT_DIR = f"runs/{SUBSET}"
MERGED_DIR = f"outputs/llama-3.2-1b-{SUBSET}"

MAX_SEQ_LENGTH = 2048
SEED = 3407

In [3]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=False,
)

==((====))==  Unsloth 2026.8.15: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4060. Num GPUs = 1. Max memory: 7.996 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-1B as a legacy tokenizer.


In [4]:
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

In [5]:
train_ds = load_dataset("json", data_files=DATA_PATH, split="train")
eval_ds = load_dataset("json", data_files=EVAL_PATH, split="train")
print(train_ds, eval_ds)

Dataset({
    features: ['text'],
    num_rows: 5000
}) Dataset({
    features: ['text'],
    num_rows: 1000
})


In [6]:
texts = list(train_ds["text"])
lens = [len(x) for x in tokenizer(texts, add_special_tokens=False)["input_ids"]]
print(np.percentile(lens, [50, 90, 95, 99, 99.9]).round(0), "max:", max(lens))
print("rows over 1024:", sum(l > 1024 for l in lens))

[ 499.  877. 1029. 1573. 1970.] max: 2041
rows over 1024: 258


In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
model.print_trainable_parameters()

Unsloth 2026.8.15 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039


In [8]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        per_device_eval_batch_size=1,
        eval_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        optim="adamw_8bit",
        weight_decay=0.01,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=200,
        save_total_limit=2,
        output_dir=OUTPUT_DIR,
        seed=SEED,
        report_to="none",
    ),
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"]:   0%|          | 0/5000 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"]:   0%|          | 0/1000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [9]:
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [10]:
gpu = torch.cuda.get_device_properties(0)
start = torch.cuda.max_memory_reserved() / 1024 ** 3
print(f"{gpu.name}, {gpu.total_memory / 1024 ** 3:.1f} GB total, {start:.2f} GB reserved")

NVIDIA GeForce RTX 4060, 8.0 GB total, 2.39 GB reserved


In [11]:
stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 1 | Total steps = 313
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
100,0.454637,0.464289
200,0.434012,0.434042
300,0.401614,0.426247
313,0.435691,0.426212


Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Unsloth: Restored added_tokens_decoder metadata in runs/5k\checkpoint-200\tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in runs/5k\checkpoint-313\tokenizer_config.json.


In [12]:
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")

Unsloth: Restored added_tokens_decoder metadata in outputs/llama-3.2-1b-5k\tokenizer_config.json.


Found HuggingFace hub cache directory: C:\Users\Olive\.cache\huggingface\hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `outputs/llama-3.2-1b-5k`: 100%|██████████| 1/1 [00:21<00:00, 21.54s/it]


Successfully copied all 1 files from cache to `outputs/llama-3.2-1b-5k`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:26<00:00, 26.72s/it]


Unsloth: Merge process complete. Saved to `D:\Llama-3.2-1B-MathCodeInstruct\Llama-3.2-1B-MathCodeInstruct-5k\outputs\llama-3.2-1b-5k`
